In [4]:
import pandas as pd

df = pd.read_csv(r"C:\Users\MOIZ\DeepLearning\microProject2\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()


review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

Pre Processing

1] Convert to lowercase

In [7]:
df["review"] = df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [8]:
import re # regular expression
#regex library

In [9]:
def remove_urls(text):
    text = re.sub(r"http\S+","",text) #(pattern,repl,string)
    return text
df["review"] = df["review"].apply(remove_urls)

Remove Punctuation

In [10]:
def remove_punctuations(text):
    text = re.sub(r"[^A-za-z0-9\s]","",text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

Removing HTML

In [11]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_html)

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")


    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")

    return text
df["review"] = df["review"].apply(remove_stopwords)
    

Steming

In [13]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words =[]
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)


Encoding

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])


In [15]:
y = df["sentiment"]

In [16]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])

In [19]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4054738 stored elements and shape (49582, 5000)>

In [20]:
print(X)    

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4054738 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3537)	0.055172586141150226
  (0, 2866)	0.09360715313699464
  (0, 4940)	0.11471517555103908
  (0, 3000)	0.4719818270486242
  (0, 1276)	0.13576311159915805
  (0, 2288)	0.049511010686691684
  (0, 1932)	0.07912207930050451
  (0, 3549)	0.0964617216417389
  (0, 1361)	0.061646058918606104
  (0, 1961)	0.06155737067674403
  (0, 220)	0.08588300923323462
  (0, 1619)	0.07386441987591297
  (0, 4367)	0.04201440342897208
  (0, 4169)	0.17809913659698137
  (0, 3692)	0.033559596604044846
  (0, 4737)	0.26797604590397783
  (0, 3804)	0.04430009216877035
  (0, 4769)	0.05877112365001594
  (0, 1738)	0.037507181147532515
  (0, 4495)	0.07613686094268304
  (0, 3856)	0.17546267791412257
  (0, 1629)	0.061429350281813636
  (0, 1861)	0.07434740009253278
  (0, 3328)	0.0640745783502313
  (0, 3331)	0.08447124957013613
  :	:
  (49581, 4890)	0.10685497349746331
  (49581, 1540)	0.17578667665

Dataset and DataLoaders

In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [32]:
import torch
from torch.utils.data import TensorDataset , DataLoader


In [33]:
X_train = X_train.toarray()
X_test=X_test.toarray()


AttributeError: 'numpy.ndarray' object has no attribute 'toarray'

In [34]:
X_train

array([[0.21326894, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(39665, 5000))

In [38]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [39]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

Build RNN

In [45]:
import torch.nn as nn
import torch.optim as optim

In [47]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size =  hidden_size
        self.num_layers =  num_layers
        #RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out, _ = self.rnn(x,h0)
        out = self.fc(out)

        out = self.fc(out[:,-1,:])
        return out 

In [48]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())
